In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['target'] = housing.target  # median house value (in $100k)

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(df.shape)        # (20640, 9)
print(df.head())

(20640, 9)
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  target  
0    -122.23   4.526  
1    -122.22   3.585  
2    -122.24   3.521  
3    -122.25   3.413  
4    -122.25   3.422  


# ** Linear Regression (OLS) No regularization **
Finds weights w that minimize Σ(yᵢ − ŷᵢ)². The solution is analytical (no gradient descent needed): w = (XᵀX)⁻¹Xᵀy. Simple, fast, fully interpretable — but sensitive to outliers and correlated features.

In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

model = LinearRegression()
model.fit(X_train_sc, y_train)

y_pred = model.predict(X_test_sc)

r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(np.mean((y_test - y_pred)**2))

print(f"R²  : {r2:.3f}")
print(f"MAE : {mae:.3f}  (~${mae*100:.0f}k off on average)")
print(f"RMSE: {rmse:.3f}")

# Coefficients — which features matter most?
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coef': model.coef_
}).sort_values('coef', key=abs, ascending=False)
print(coef_df)

R²  : 0.576
MAE : 0.533  (~$53k off on average)
RMSE: 0.746
      feature      coef
6    Latitude -0.896929
7   Longitude -0.869842
0      MedInc  0.854383
3   AveBedrms  0.339259
2    AveRooms -0.294410
1    HouseAge  0.122546
5    AveOccup -0.040829
4  Population -0.002308


#**Ridge Regression (L2) Shrinks all coefficients**
Minimizes Σ(yᵢ − ŷᵢ)² + α·Σwᵢ². The α penalty shrinks all coefficients toward zero uniformly — no feature is fully eliminated. Ideal when many features are mildly useful and correlated (like Latitude/Longitude here).

In [3]:
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

# Find best alpha automatically with cross-validation
alphas = [0.01, 0.1, 1, 10, 100, 1000]
model_cv = RidgeCV(alphas=alphas, cv=5)
model_cv.fit(X_train_sc, y_train)
print(f"Best alpha: {model_cv.alpha_}")

model = Ridge(alpha=model_cv.alpha_)
model.fit(X_train_sc, y_train)
y_pred = model.predict(X_test_sc)

r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(np.mean((y_test - y_pred)**2))

print(f"R²  : {r2:.3f}")
print(f"MAE : {mae:.3f}  (~${mae*100:.0f}k off on average)")
print(f"RMSE: {rmse:.3f}")

coef_df = pd.DataFrame({
    'feature': X.columns,
    'coef': model.coef_
}).sort_values('coef', key=abs, ascending=False)
print(coef_df)

Best alpha: 0.01
R²  : 0.576
MAE : 0.533  (~$53k off on average)
RMSE: 0.746
      feature      coef
6    Latitude -0.896921
7   Longitude -0.869834
0      MedInc  0.854382
3   AveBedrms  0.339257
2    AveRooms -0.294408
1    HouseAge  0.122547
5    AveOccup -0.040829
4  Population -0.002307


In [4]:
from sklearn.linear_model import Lasso, LassoCV
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

# Cross-validated alpha search
model_cv = LassoCV(cv=5, random_state=42, max_iter=5000)
model_cv.fit(X_train_sc, y_train)
print(f"Best alpha: {model_cv.alpha_:.4f}")

model = Lasso(alpha=model_cv.alpha_, max_iter=5000)
model.fit(X_train_sc, y_train)
y_pred = model.predict(X_test_sc)

r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(np.mean((y_test - y_pred)**2))

print(f"R²  : {r2:.3f}")
print(f"MAE : {mae:.3f}  (~${mae*100:.0f}k off on average)")
print(f"RMSE: {rmse:.3f}")

# Features Lasso KEPT vs zeroed out
coef_df = pd.DataFrame({'feature': X.columns, 'coef': model.coef_})
print("Features kept:", coef_df[coef_df.coef != 0]['feature'].tolist())
print("Features removed:", coef_df[coef_df.coef == 0]['feature'].tolist())

Best alpha: 0.0008
R²  : 0.577
MAE : 0.533  (~$53k off on average)
RMSE: 0.745
Features kept: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Features removed: []


What is regularization?
what is L1 and L2 regularization?

#Task 1 —
Regression: Predict used car prices (fetch from a URL / seaborn's mpg dataset as proxy)
The data has missing values, categorical columns, outliers in price/mileage, and skewed distributions. You'll need to clean, engineer features, then predict price.
#Task 2 —
Classification: Predict heart disease (sklearn.datasets.load_heart_disease / UCI Heart Disease)
Mixed types, some columns with ? as nulls, class imbalance, and features that need scaling + encoding before any model works.